# Survey Response Analysis

## Project Overview
Analyze survey responses using descriptive statistics, segment comparisons, missingness patterns, and response bias considerations.

## Learning Objectives
- Summarize Likert-scale and categorical survey data properly
- Analyze response patterns across demographic segments
- Detect and visualize missingness and non-response bias
- Interpret survey results with appropriate caveats

## Problem Statement
Organizations that run surveys need to understand what the data actually says — beyond simple averages — including who responded, who didn't, and whether results are representative.

## Why This Project Matters
Surveys drive decisions across business, government, and academia. Poor analysis leads to false conclusions. Understanding response bias, missingness, and segment differences is critical for valid interpretation.

## Dataset Overview
Synthetic employee engagement survey: ~1,200 responses with 8 Likert-scale questions, demographics, and deliberately introduced missingness patterns.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 1200
dept = np.random.choice(['Engineering', 'Sales', 'Marketing', 'HR', 'Operations', 'Finance'], n,
                          p=[0.25, 0.20, 0.15, 0.10, 0.18, 0.12])
tenure = np.random.choice(['<1yr', '1-3yr', '3-5yr', '5-10yr', '10+yr'], n,
                            p=[0.15, 0.25, 0.25, 0.20, 0.15])
level = np.random.choice(['Junior', 'Mid', 'Senior', 'Manager'], n, p=[0.30, 0.35, 0.22, 0.13])
age_group = np.random.choice(['18-25', '26-35', '36-45', '46-55', '55+'], n, p=[0.12, 0.30, 0.28, 0.20, 0.10])

# Likert questions (1-5)
questions = ['Q1_Satisfaction', 'Q2_WorkLifeBalance', 'Q3_Management', 'Q4_Growth',
             'Q5_Compensation', 'Q6_TeamCulture', 'Q7_Recognition', 'Q8_Recommend']
base_sentiment = np.random.normal(3.5, 0.8, n)  # overall sentiment driver

responses = {}
for q in questions:
    q_noise = np.random.normal(0, 0.6, n)
    q_bias = np.random.uniform(-0.3, 0.3)
    vals = np.clip(np.round(base_sentiment + q_noise + q_bias), 1, 5).astype(int)
    responses[q] = vals

df = pd.DataFrame({
    'ResponseID': [f'R{i:05d}' for i in range(n)],
    'Department': dept, 'Tenure': tenure, 'Level': level, 'AgeGroup': age_group,
    **responses
})

# Introduce MNAR missingness: disengaged people skip Q7 and Q8 more
for q in ['Q7_Recognition', 'Q8_Recommend']:
    miss_prob = np.where(base_sentiment < 3.0, 0.25, 0.05)
    miss_mask = np.random.binomial(1, miss_prob, n).astype(bool)
    df.loc[miss_mask, q] = np.nan

# Random MCAR missingness on other questions
for q in ['Q3_Management', 'Q5_Compensation']:
    miss_mask = np.random.binomial(1, 0.05, n).astype(bool)
    df.loc[miss_mask, q] = np.nan

print(f'Dataset: {df.shape}')
print(f'Overall missing: {df.isnull().sum().sum()} values ({df.isnull().sum().sum()/(n*len(questions)):.1%} of question data)')
df.head()

## Data Validation Checks

In [ ]:
print('Missing values per question:')
print(df[questions].isnull().sum())
print(f'\nComplete responses: {df.dropna().shape[0]} / {len(df)} ({df.dropna().shape[0]/len(df):.1%})')

## Missingness Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
miss_pct = df[questions].isnull().mean() * 100
miss_pct.plot.barh(ax=axes[0], edgecolor='black', color='coral')
axes[0].set_title('Missing Response Rate by Question')
axes[0].set_xlabel('% Missing')

# Missingness by department
dept_miss = df.groupby('Department')[questions].apply(lambda x: x.isnull().mean().mean() * 100)
dept_miss.sort_values().plot.barh(ax=axes[1], edgecolor='black', color='steelblue')
axes[1].set_title('Avg Missing Rate by Department')
axes[1].set_xlabel('% Missing')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'missingness.png'), dpi=100, bbox_inches='tight')
plt.show()

## Overall Response Summary

In [ ]:
summary = df[questions].describe().T[['mean', 'std', '50%']].round(2)
summary.columns = ['Mean', 'Std', 'Median']
summary = summary.sort_values('Mean', ascending=False)
print('Question Summary (1-5 Likert scale):')
print(summary)

fig, ax = plt.subplots(figsize=(10, 6))
summary['Mean'].plot.barh(ax=ax, xerr=summary['Std'], edgecolor='black', color='teal', capsize=3)
ax.set_title('Mean Response ± Std by Question')
ax.set_xlabel('Mean Score (1-5)')
ax.axvline(3, color='red', linestyle='--', alpha=0.5, label='Neutral (3)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'response_summary.png'), dpi=100, bbox_inches='tight')
plt.show()

## Response Distribution per Question

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, q in zip(axes.flat, questions):
    counts = df[q].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
    counts.plot.bar(ax=ax, edgecolor='black', color=['#d73027','#fc8d59','#fee090','#91bfdb','#4575b4'])
    ax.set_title(q.split('_', 1)[1], fontsize=9)
    ax.tick_params(axis='x', rotation=0)
    ax.set_xlabel('')
plt.suptitle('Response Distribution per Question', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'response_distributions.png'), dpi=100, bbox_inches='tight')
plt.show()

## Segment Comparison: Department

In [ ]:
dept_means = df.groupby('Department')[questions].mean().round(2)
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(dept_means, annot=True, fmt='.2f', cmap='RdYlGn', center=3, ax=ax, vmin=2, vmax=5)
ax.set_title('Mean Response by Department')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'department_heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Segment Comparison: Tenure

In [ ]:
tenure_order = ['<1yr', '1-3yr', '3-5yr', '5-10yr', '10+yr']
tenure_means = df.groupby('Tenure')[questions].mean().reindex(tenure_order).round(2)
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(tenure_means, annot=True, fmt='.2f', cmap='RdYlGn', center=3, ax=ax, vmin=2, vmax=5)
ax.set_title('Mean Response by Tenure')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'tenure_heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Engagement Score Composite

In [ ]:
df['EngagementScore'] = df[questions].mean(axis=1).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['EngagementScore'].hist(bins=20, ax=axes[0], edgecolor='black', color='steelblue')
axes[0].set_title('Composite Engagement Score Distribution')
axes[0].set_xlabel('Score (1-5)')
axes[0].axvline(df['EngagementScore'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["EngagementScore"].mean():.2f}')
axes[0].legend()

sns.boxplot(data=df, x='Level', y='EngagementScore',
            order=['Junior', 'Mid', 'Senior', 'Manager'], ax=axes[1])
axes[1].set_title('Engagement by Level')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'engagement_score.png'), dpi=100, bbox_inches='tight')
plt.show()

## Non-Response Bias Check

In [ ]:
# Compare respondents who completed Q8 vs those who didn't
q8_complete = df[df['Q8_Recommend'].notna()]
q8_missing = df[df['Q8_Recommend'].isna()]
print(f'Q8 completers: {len(q8_complete)}, Q8 non-responders: {len(q8_missing)}')
print(f'\nMean engagement (completers): {q8_complete["EngagementScore"].mean():.2f}')
print(f'Mean engagement (non-responders): {q8_missing["EngagementScore"].mean():.2f}')
print(f'\nThis {q8_missing["EngagementScore"].mean() - q8_complete["EngagementScore"].mean():+.2f} gap suggests MNAR bias:')
print('Disengaged employees are more likely to skip the recommendation question.')

## Interpretation and Business Insight
- **Overall engagement** averages around 3.5/5 — moderately positive but with room for improvement
- **Recognition and recommendation** questions have the highest missingness — AND the missing responses skew toward disengaged employees (MNAR bias)
- **Naive analysis** (ignoring missing data) would overestimate engagement by ~0.1-0.2 points
- **Department differences** are modest but actionable — specific teams may need targeted interventions
- **Tenure effects** may reveal engagement lifecycle patterns (honeymoon → disillusionment → adaptation)
- **Compensation** consistently scores lowest — a common finding in engagement surveys

## Limitations
- Synthetic data with simplified response model
- No free-text responses
- No response timing data
- No longitudinal tracking (single survey wave)
- No external validation of engagement scores

## How to Improve This Project
- Add multiple survey waves for trend analysis
- Include free-text sentiment analysis
- Build response propensity models to weight for non-response bias
- Add benchmark comparisons against industry norms
- Include pulse survey data between annual surveys

## Production Considerations
- Automated survey analysis pipelines
- Non-response bias adjustment in reported results
- Confidentiality-preserving reporting at small group sizes
- Manager-facing dashboards with action recommendations

## Common Mistakes
- Reporting means without acknowledging missingness patterns
- Treating Likert data as ratio-scale (means of ordinal data have caveats)
- Ignoring that low response rates may make results unrepresentative
- Drawing causal conclusions from cross-sectional survey data

## Mini Challenge / Exercises
1. Which department has the highest composite engagement score?
2. Calculate the response rate for each question. Which has the lowest?
3. If you impute missing Q8 values with the respondent's average of other questions, how does the Q8 mean change?
4. Is there a statistically significant difference in engagement between Managers and Juniors? (Hint: use a t-test.)

## Final Summary / Key Takeaways
- Survey analysis requires attention to missingness, not just averages
- Non-response bias (MNAR) is common in engagement surveys — disengaged employees skip sensitive questions
- Segment comparisons reveal actionable differences across departments, tenure, and levels
- Composite engagement scores simplify communication but mask question-level variation
- Honest reporting of limitations and biases builds trust in survey programs